In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yt
import math
from matplotlib import ticker
from colormap import rgb2hex                         # keep if you prefer it
# NEW ────────────────────────────────────────────────────────────────────────
# axes_grid → axes_grid1   (same functionality, maintained sub-package)
from mpl_toolkits.axes_grid1.inset_locator import (
    inset_axes, InsetPosition, mark_inset
)
# ────────────────────────────────────────────────────────────────────────────

# --------------------------------------------------------------------------
# Matplotlib global style
# --------------------------------------------------------------------------
plt.rc("text", usetex=False)
plt.rc("font", size=12)
plt.rc("axes", titlesize=60)
plt.rcParams.update({
    "legend.fontsize": 12,
    "lines.markersize": 5,
    "lines.markeredgecolor": "k",
    "lines.markeredgewidth": 0.2,
    "lines.linewidth": 1.0,
})

formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(False)
formatter.set_powerlimits((-1, 1))

# --------------------------------------------------------------------------
# Plot parameters
# --------------------------------------------------------------------------
aspect = 1.0

# Custom colour palette (red → blue gradient)
ncol = 5
red  = np.array([254,   0,   2], dtype=int)
blue = np.array([  3,   2, 252], dtype=int)

c1 = np.linspace(red[0],  blue[0],  ncol, dtype=int)
c2 = np.linspace(red[1],  blue[1],  ncol, dtype=int)
c3 = np.linspace(red[2],  blue[2],  ncol, dtype=int)

colors = [rgb2hex(r, g, b) for r, g, b in zip(c1, c2, c3)][::-1]

markers = ["o", "^", "v", "s", "D", "X", "*"]

In [ ]:
def loadtxt_fortran(filename):
    with open(filename, 'r') as f:
        content = f.read().replace('D', 'E')
    from io import StringIO
    return np.loadtxt(StringIO(content))

In [ ]:
def beta_0(alpha):
    return ((1+alpha)/2)*(1-((1-alpha)/3))

def zeta_0(alpha):
    return (1-alpha**2)*(5/12)

def a_steady(beta, zeta):
    return (np.sqrt((3*zeta)/(2*beta)))*(beta+zeta)

In [ ]:
def detect_plateau(tg, window=100, threshold=1e-3):
    rel_change = np.abs(np.diff(tg) / tg[:-1])
    idx = np.where(rel_change < threshold)[0]

    # Find longest consecutive plateau
    from itertools import groupby
    from operator import itemgetter
    groups = [list(map(itemgetter(1), g)) for k, g in groupby(enumerate(idx), lambda ix: ix[0]-ix[1])]
    if not groups:
        raise RuntimeError("No steady state detected.")
    plateau = max(groups, key=len)
    
    if len(plateau) < window:
        print("Warning: short steady region found.")
    
    return plateau[0], plateau[-1]


In [ ]:
# Chi function (Ma and Ahmadi)
def chi_ma(sol_frac):
    sol_frac_pack = 0.643
    C1, C2, C3, C_POW = 2.5, 4.5904, 4.515439, 0.67802
    if sol_frac > sol_frac_pack:
        return 1000.0
    numer = 1.0 + C1 * sol_frac + C2 * sol_frac**2 + C3 * sol_frac**3
    numer = numer * 4.0 * sol_frac
    denom = (1.0 - (sol_frac/sol_frac_pack)**3.0)**C_POW
    chi = 1.0 + numer/denom
    return min(chi, 1000.0)

def chi_Carnahan_Starling(solfrac):
    return (1 - solfrac/2)/((1-solfrac)**3)

L = 0.1
sigma = 0.01
sol_frac = 0.01
n = (6.0 * sol_frac) / (np.pi * sigma**3)

chi_val = chi_ma(sol_frac)
chi_val

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

# Paths
base_dir = '/home/muhammed/Documents/Thesis/Shear_Simulation_Results/Run_shear_NN'
alpha_dirs = ['60', '65', '70', '75', '80', '85', '90', '95', '100']  # Skip '100'

# Simulation constants
n_particles = 100000  # Adjust if needed
volume = 0.1**3       # Assuming your domain is 0.1x0.1x0.1
density = n_particles / volume

# Store results
alpha_vals = []
Pxx_vals = []
Pyy_vals = []
Pxy_vals = []
Pzz_vals = []
psi1_vals = []
psi2_vals = []


for folder in alpha_dirs:
    folder_path = os.path.join(base_dir, folder)
    
    # Load data
    tg = np.loadtxt(os.path.join(folder_path, 'tg.txt'))[:,2]  # Assuming format: time, Tg
    pijk = loadtxt_fortran(os.path.join(folder_path, 'Pijk.txt'))
    pijc = loadtxt_fortran(os.path.join(folder_path, 'Pijc.txt'))

    # Sum kinetic + collisional
    Pxx = pijk[1:,0] + pijc[:,0]
    Pyy = pijk[1:,1] + pijc[:,3]
    Pxy = pijk[1:,3] + pijc[:,1]
    Pzz = pijk[1:,2] + pijc[:,5]

    # Find steady state
    steady_start, steady_end = detect_plateau(tg)
    
    # Average over steady state window
    Pxx_mean = np.mean(Pxx[-500:])
    Pyy_mean = np.mean(Pyy[-500:])
    Pxy_mean = np.mean(Pxy[-500:])
    Pzz_mean = np.mean(Pzz[-500:])
    Tg_mean  = np.mean(tg[-500:])
    
    # Compute reduced stresses
    Pxx_star = Pxx_mean / (density * Tg_mean)
    Pyy_star = Pyy_mean / (density * Tg_mean)
    Pzz_star = Pzz_mean / (density * Tg_mean)
    Pxy_star = Pxy_mean / (density * Tg_mean)
    press = (Pxx_star + Pyy_star + Pzz_star)/3.0
    psi_1 = (Pyy_star - Pxx_star) / press if press > 1e-10 else 0.0
    psi_2 = (Pxx_star - Pzz_star) / press if press > 1e-10 else 0.0

    # Store
    alpha = int(folder) / 100.0
    alpha_vals.append(alpha)
    Pxx_vals.append(Pxx_star)
    Pyy_vals.append(Pyy_star)
    Pxy_vals.append(Pxy_star)
    Pzz_vals.append(Pzz_star)
    psi1_vals.append(psi_1)
    psi2_vals.append(psi_2)

# Convert to arrays
alpha_vals = np.array(alpha_vals)
Pxx_vals = np.array(Pxx_vals)
Pyy_vals = np.array(Pyy_vals)
Pxy_vals = np.array(Pxy_vals)
Pzz_vals = np.array(Pzz_vals)
psi1_vals = np.array(psi1_vals)
psi2_vals = np.array(psi2_vals)

alphas = np.linspace(0.5, 1, 1000)
betas = beta_0(alphas)
zetas = zeta_0(alphas)
Pxx_KT = (betas + 3*zetas)/(betas + zetas)
Pyy_KT = betas/(betas+zetas)
Pxy_KT = -(betas/((betas+zetas)**2)) * a_steady(betas, zetas)


# Plotting
plt.figure(figsize=(8,6))
plt.scatter(alpha_vals, Pyy_vals*1000, marker='o', label=r'$P_{xx}^*$')
plt.plot(alphas, Pyy_KT, '-r')
plt.scatter(alpha_vals, Pxx_vals*1000, marker='*', label=r'$P_{yy}^*$')
plt.plot(alphas, Pxx_KT, '-k')
plt.scatter(alpha_vals, Pxy_vals*1000, marker='^', label=r'$P_{xy}^*$')
plt.plot(alphas, Pxy_KT, '-b')

plt.scatter(alpha_vals, Pzz_vals*1000, marker='H', label=r'$P_{zz}^*$')
plt.xlabel(r'Coefficient of Restitution $\alpha$')
plt.ylabel(r'Reduced Stress $P_{ij}^*$')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# log (alpha=0.9)
fig,ax = plt.subplots();
ax.yaxis.set_major_formatter(formatter)

ax.scatter(alpha_vals, Pyy_vals*1000, label=r'$P_{xx}^*$', marker=markers[0], color=colors[0])
ax.scatter(alpha_vals, Pxx_vals*1000, label=r'$P_{yy}^*$', marker=markers[1], color=colors[1])
ax.scatter(alpha_vals, Pzz_vals*1000, label=r'$P_{zz}^*$', marker=markers[2], color=colors[2])
ax.scatter(alpha_vals, Pxy_vals*1000, label=r'$P_{xy}^*$', marker=markers[3], color=colors[3])

ax.plot(alphas, Pyy_KT, '-', color=colors[0])
ax.plot(alphas, Pxx_KT, '-', color=colors[1])
ax.plot(alphas, Pxy_KT, '-', color=colors[3])

ax.set_xlim([0.55,1.02])
ax.set_ylim([-0.6,1.75])
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel(r'$P_{ij}^*$',labelpad=-4)
xval, yval = ax.get_xlim(), ax.get_ylim()
xrange = xval[1]-xval[0];
yrange = yval[1]-yval[0];
ax.set_aspect(aspect*(xrange/yrange), adjustable='box')

ax.tick_params(axis='both',direction='in',which='both',right=True,top=True)
ax.minorticks_on()
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.15, 0.5),  # X, Y: lower values shift it down
    fontsize=10,
    frameon=True
)

mng = plt.get_current_fig_manager()
mng.full_screen_toggle()
plt.savefig('ScaledTClasslog09.eps',bbox_inches='tight')